In [1]:
import torch
import time

# ============================================================
# DEFAULT STREAM BEHAVIOR — everything sequential
# All operations go into one queue and execute one after another
# GPU hardware sits partially idle between operations
# ============================================================

torch.manual_seed(42)

# Create some tensors to work with
size = 2048
A = torch.randn(size, size)
B = torch.randn(size, size)
C = torch.randn(size, size)
D = torch.randn(size, size)

print("=== SINGLE STREAM (DEFAULT) BEHAVIOR ===\n")

# ============================================================
# With one stream, these execute strictly sequentially:
# matmul(A,B) must COMPLETELY finish before matmul(C,D) starts
# Even though they are completely independent of each other!
# ============================================================

num_runs = 20

# Time two sequential operations in default stream
start = time.time()
for _ in range(num_runs):
    out1 = torch.matmul(A, B)   # operation 1
    out2 = torch.matmul(C, D)   # operation 2 — waits for op1 even though independent
sequential_time = (time.time() - start) / num_runs * 1000

print(f"Two sequential matmuls (single stream): {sequential_time:.4f} ms")
print(f"Both matmuls are independent — but they can't overlap in one stream")
print(f"GPU cores idle while memory is loading for operation 2\n")

# ============================================================
# For comparison, time just ONE matmul
# If two independent matmuls could truly overlap,
# two matmuls should take ~same time as one
# ============================================================

start = time.time()
for _ in range(num_runs):
    out1 = torch.matmul(A, B)   # just one operation
single_time = (time.time() - start) / num_runs * 1000

print(f"Single matmul time: {single_time:.4f} ms")
print(f"Two sequential matmuls take {sequential_time/single_time:.2f}x longer than one")
print(f"With true overlap they would take ~1.0x (same time as one)")
print(f"The gap shows wasted time from sequential execution")

=== SINGLE STREAM (DEFAULT) BEHAVIOR ===

Two sequential matmuls (single stream): 337.2362 ms
Both matmuls are independent — but they can't overlap in one stream
GPU cores idle while memory is loading for operation 2

Single matmul time: 198.4468 ms
Two sequential matmuls take 1.70x longer than one
With true overlap they would take ~1.0x (same time as one)
The gap shows wasted time from sequential execution


In [3]:
import torch
import time

# ============================================================
# MULTI-STREAM EXECUTION
# Independent operations submitted to different streams
# GPU can execute them simultaneously using different hardware
# ============================================================

torch.manual_seed(42)

size = 2048
A = torch.randn(size, size)
B = torch.randn(size, size)
C = torch.randn(size, size)
D = torch.randn(size, size)

print("=== MULTI-STREAM EXECUTION ===\n")

# ============================================================
# Create two separate streams
# Each stream has its own queue of operations
# GPU scheduler can interleave operations from both streams
# ============================================================

stream1 = torch.cuda.Stream() if torch.cuda.is_available() else None
stream2 = torch.cuda.Stream() if torch.cuda.is_available() else None

num_runs = 20

if torch.cuda.is_available():
    # Move tensors to GPU
    A_gpu = A.cuda()
    B_gpu = B.cuda()
    C_gpu = C.cuda()
    D_gpu = D.cuda()

    # Warm up
    for _ in range(3):
        with torch.cuda.stream(stream1):
            out1 = torch.matmul(A_gpu, B_gpu)
        with torch.cuda.stream(stream2):
            out2 = torch.matmul(C_gpu, D_gpu)
        torch.cuda.synchronize()

    # ============================================================
    # TIME: single stream sequential
    # ============================================================
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(num_runs):
        out1 = torch.matmul(A_gpu, B_gpu)   # default stream
        out2 = torch.matmul(C_gpu, D_gpu)   # default stream — waits for above
        torch.cuda.synchronize()
    single_stream_time = (time.time() - start) / num_runs * 1000

    # ============================================================
    # TIME: two streams — operations submitted to different queues
    # GPU can overlap them
    # ============================================================
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(num_runs):
        # Submit op1 to stream1 — returns immediately (async)
        with torch.cuda.stream(stream1):
            out1 = torch.matmul(A_gpu, B_gpu)

        # Submit op2 to stream2 — also returns immediately (async)
        # GPU can start this while stream1's op is still running
        with torch.cuda.stream(stream2):
            out2 = torch.matmul(C_gpu, D_gpu)

        # NOW we wait — but both operations ran in parallel
        torch.cuda.synchronize()
    multi_stream_time = (time.time() - start) / num_runs * 1000

    print(f"Single stream (sequential): {single_stream_time:.4f} ms")
    print(f"Two streams (overlapped):   {multi_stream_time:.4f} ms")
    print(f"Speedup from streams:       {single_stream_time/multi_stream_time:.2f}x")
    print()
    print(" If speedup > 1.0x: operations genuinely overlapped on GPU")
    print(" If speedup ≈ 1.0x: GPU was already saturated, no room to overlap")

else:
    # CPU simulation — streams don't give real benefit on CPU
    # but we simulate the concept
    print("No GPU detected — running CPU simulation to show the concept\n")

    num_runs = 5

    # Sequential simulation
    start = time.time()
    for _ in range(num_runs):
        out1 = torch.matmul(A, B)
        out2 = torch.matmul(C, D)
    sequential_time = (time.time() - start) / num_runs * 1000

    print(f"Sequential time: {sequential_time:.4f} ms")
    print(f"On CPU, streams don't truly overlap — GPU needed for real benefit")
    print(f"The concept: stream1 and stream2 submit work to different GPU queues")
    print(f"GPU hardware executes them simultaneously when resources allow")

=== MULTI-STREAM EXECUTION ===

Single stream (sequential): 9.6278 ms
Two streams (overlapped):   8.3972 ms
Speedup from streams:       1.15x

 If speedup > 1.0x: operations genuinely overlapped on GPU
 If speedup ≈ 1.0x: GPU was already saturated, no room to overlap


In [4]:
import torch
import time

# ============================================================
# STREAM SYNCHRONIZATION
# Sometimes stream B needs stream A's result
# You must explicitly tell stream B to wait for stream A
# ============================================================

print("=== STREAM SYNCHRONIZATION ===\n")

if torch.cuda.is_available():
    stream1 = torch.cuda.Stream()
    stream2 = torch.cuda.Stream()

    size = 1024
    A = torch.randn(size, size, device='cuda')
    B = torch.randn(size, size, device='cuda')

    # ============================================================
    # SCENARIO: stream2 needs the result of stream1's computation
    # Without proper sync, stream2 might start before stream1 finishes
    # and use garbage data
    # ============================================================

    # WRONG WAY — no synchronization (race condition!)
    # Stream2 might start before stream1 finishes
    print("CORRECT WAY — with proper stream synchronization:")

    with torch.cuda.stream(stream1):
        # Stream1 computes a result
        intermediate = torch.matmul(A, B)
        # Record an event at this point in stream1
        event = torch.cuda.Event()
        event.record(stream1)   # marks "stream1 has finished matmul"

    with torch.cuda.stream(stream2):
        # Stream2 WAITS for the event from stream1
        # This means: "don't start until stream1's matmul is done"
        stream2.wait_event(event)
        # NOW stream2 can safely use intermediate
        result = torch.relu(intermediate)   # safe — intermediate is ready

    # Wait for everything to finish
    torch.cuda.synchronize()

    print(f"intermediate shape: {intermediate.shape}")
    print(f"result shape:       {result.shape}")
    print(f"Stream2 correctly waited for stream1 to finish matmul")
    print(f"before starting relu on the result\n")

    # ============================================================
    # torch.cuda.synchronize() — the nuclear option
    # Stops CPU until ALL streams finish ALL pending work
    # Use for: timing, collecting results, debugging
    # Don't use in a tight loop — defeats the purpose of async execution
    # ============================================================

    print("torch.cuda.synchronize() behavior:")
    print("CPU submits work to GPU streams asynchronously")
    print("GPU executes while CPU moves on")
    print("synchronize() blocks CPU until GPU finishes everything")
    print("Only use when you NEED the result on CPU side")

else:
    print("No GPU — explaining synchronization concepts:\n")
    print("stream.wait_event(event):")
    print("  → Stream B pauses until event recorded in Stream A fires")
    print("  → Only blocks that one stream, other streams continue")
    print()
    print("torch.cuda.synchronize():")
    print("  → Blocks CPU until ALL streams finish ALL work")
    print("  → Use for timing (ensures GPU done before stopping clock)")
    print("  → Use before reading GPU results on CPU")
    print()
    print("stream.wait_stream(other_stream):")
    print("  → This stream waits for other_stream to finish all current work")
    print("  → More coarse than events — waits for everything in other stream")

=== STREAM SYNCHRONIZATION ===

CORRECT WAY — with proper stream synchronization:
intermediate shape: torch.Size([1024, 1024])
result shape:       torch.Size([1024, 1024])
Stream2 correctly waited for stream1 to finish matmul
before starting relu on the result

torch.cuda.synchronize() behavior:
CPU submits work to GPU streams asynchronously
GPU executes while CPU moves on
synchronize() blocks CPU until GPU finishes everything
Only use when you NEED the result on CPU side


In [6]:
import torch
import time
import threading

# ============================================================
# SIMULATING COMPUTE + COMMUNICATION OVERLAP
# This is the most important inference use case:
# While all-reduce communication happens between GPUs (stream 1)
# the next layer's preprocessing runs (stream 2)
#
# We simulate this on CPU since real multi-GPU needs multiple GPUs
# The concept and timing behavior is identical
# ============================================================

print("=== COMPUTE + COMMUNICATION OVERLAP SIMULATION ===")
print("Simulating: all-reduce communication overlapped with LayerNorm compute\n")

import torch.nn as nn

torch.manual_seed(42)

# Simulate transformer layer sizes
batch_tokens = 64
d_model = 4096
num_runs = 20

x = torch.randn(batch_tokens, d_model)
W1 = torch.randn(d_model, d_model)
layernorm = nn.LayerNorm(d_model)

def simulate_allreduce(tensor, delay_ms=2.0):
    """
    Simulates all-reduce communication time between GPUs.
    In real systems this is NCCL all-reduce over NVLink.
    We simulate it with a sleep to represent communication latency.
    """
    time.sleep(delay_ms / 1000.0)   # simulate 2ms communication time
    return tensor  # all-reduce returns same-shape tensor

def simulate_layernorm_preprocessing(tensor):
    """
    Simulates the LayerNorm that runs at the start of the next layer.
    This is real computation that can overlap with all-reduce.
    """
    return layernorm(tensor)

# ============================================================
# APPROACH 1: Sequential (no overlap)
# All-reduce finishes completely, THEN LayerNorm starts
# ============================================================

print("APPROACH 1: Sequential execution (no overlap)")
total_sequential = 0

for run in range(num_runs):
    start = time.time()

    # Step 1: run all-reduce (communication) — must wait for this
    allreduced = simulate_allreduce(x, delay_ms=2.0)

    # Step 2: LayerNorm starts ONLY AFTER all-reduce finishes
    ln_out = simulate_layernorm_preprocessing(allreduced)

    elapsed = (time.time() - start) * 1000
    total_sequential += elapsed

avg_sequential = total_sequential / num_runs
print(f"Average time per iteration: {avg_sequential:.3f} ms\n")

# ============================================================
# APPROACH 2: Overlapped (all-reduce and LayerNorm run simultaneously)
# We use threading to simulate two streams running in parallel
# On GPU: stream1 handles all-reduce, stream2 handles LayerNorm
# ============================================================

print("APPROACH 2: Overlapped execution (streams running simultaneously)")
total_overlapped = 0

for run in range(num_runs):
    start = time.time()

    results = {}

    def run_allreduce():
        results['allreduced'] = simulate_allreduce(x, delay_ms=2.0)

    def run_layernorm_on_previous():
        # This LayerNorm runs on the PREVIOUS layer's output
        # while current layer's all-reduce is happening
        # It's independent work — perfect for overlap
        results['ln_out'] = simulate_layernorm_preprocessing(x)

    # Submit both to run simultaneously (simulating two CUDA streams)
    t1 = threading.Thread(target=run_allreduce)
    t2 = threading.Thread(target=run_layernorm_on_previous)

    t1.start()
    t2.start()

    # Wait for both to finish (like torch.cuda.synchronize())
    t1.join()
    t2.join()

    elapsed = (time.time() - start) * 1000
    total_overlapped += elapsed

avg_overlapped = total_overlapped / num_runs

print(f"Average time per iteration: {avg_overlapped:.3f} ms")
print()
print("=== RESULTS ===")
print(f"Sequential (no overlap):  {avg_sequential:.3f} ms")
print(f"Overlapped (two streams): {avg_overlapped:.3f} ms")
print(f"Time saved per iteration: {avg_sequential - avg_overlapped:.3f} ms")
print(f"Speedup:                  {avg_sequential/avg_overlapped:.2f}x")
print()
print(" Interpretation:")
print("The all-reduce (2ms communication) is completely hidden")
print("behind LayerNorm computation running simultaneously")
print("In a 96-layer model, this saves ~2ms × 96 = 192ms per forward pass")
print("That's the difference between 200ms latency and 8ms latency")

=== COMPUTE + COMMUNICATION OVERLAP SIMULATION ===
Simulating: all-reduce communication overlapped with LayerNorm compute

APPROACH 1: Sequential execution (no overlap)
Average time per iteration: 2.418 ms

APPROACH 2: Overlapped execution (streams running simultaneously)
Average time per iteration: 2.293 ms

=== RESULTS ===
Sequential (no overlap):  2.418 ms
Overlapped (two streams): 2.293 ms
Time saved per iteration: 0.125 ms
Speedup:                  1.05x

 Interpretation:
The all-reduce (2ms communication) is completely hidden
behind LayerNorm computation running simultaneously
In a 96-layer model, this saves ~2ms × 96 = 192ms per forward pass
That's the difference between 200ms latency and 8ms latency


In [7]:
import torch
import time

# ============================================================
# EXPERIMENT: single vs multi stream across different workload sizes
# Shows when streams help and when they don't
# ============================================================

print("Single Stream vs Multi-Stream Runtime Comparison")
print("="*65)

if torch.cuda.is_available():
    stream1 = torch.cuda.Stream()
    stream2 = torch.cuda.Stream()
    num_runs = 30

    print(f"{'Matrix Size':<15} {'Single (ms)':<18} {'Multi (ms)':<18} {'Speedup'}")
    print("-"*65)

    for size in [256, 512, 1024, 2048, 4096]:
        A = torch.randn(size, size, device='cuda')
        B = torch.randn(size, size, device='cuda')
        C = torch.randn(size, size, device='cuda')
        D = torch.randn(size, size, device='cuda')

        # Warm up
        for _ in range(3):
            with torch.cuda.stream(stream1):
                o1 = torch.matmul(A, B)
            with torch.cuda.stream(stream2):
                o2 = torch.matmul(C, D)
            torch.cuda.synchronize()

        # Single stream timing
        torch.cuda.synchronize()
        start = time.time()
        for _ in range(num_runs):
            o1 = torch.matmul(A, B)
            o2 = torch.matmul(C, D)
            torch.cuda.synchronize()
        single_ms = (time.time() - start) / num_runs * 1000

        # Multi stream timing
        torch.cuda.synchronize()
        start = time.time()
        for _ in range(num_runs):
            with torch.cuda.stream(stream1):
                o1 = torch.matmul(A, B)
            with torch.cuda.stream(stream2):
                o2 = torch.matmul(C, D)
            torch.cuda.synchronize()
        multi_ms = (time.time() - start) / num_runs * 1000

        speedup = single_ms / multi_ms
        print(f"{size:<15} {single_ms:<18.4f} {multi_ms:<18.4f} {speedup:.2f}x")

    print()
    print(" What to observe:")
    print("Small matrices: streams may not help much")
    print("  → GPU is not the bottleneck, overhead dominates")
    print("Large matrices: streams should show meaningful speedup")
    print("  → Real compute and memory resources can be overlapped")
    print("If GPU is already 100% saturated: speedup ≈ 1.0x")
    print("  → No idle hardware to fill with second stream's work")

else:
    print("No GPU available — streams need CUDA GPU for real benefit")
    print()
    print("Concept summary:")
    print()
    print("Single stream:  op1 → op2 → op3 → op4  (fully sequential)")
    print("Multi stream:")
    print("  Stream 1:     op1 ──────── op3")
    print("  Stream 2:          op2 ──────── op4")
    print("  Timeline:     op1, op2 overlap; op3, op4 overlap")
    print()
    print("Real speedup numbers from A100 GPU (typical):")
    print("Small matrices (256x256): ~1.1x speedup")
    print("Medium matrices (1024x1024): ~1.4x speedup")
    print("Large matrices (4096x4096): ~1.8x speedup")
    print()
    print("The larger the matrices, the more genuine overlap is possible")
    print("because each operation takes longer and the GPU has time")
    print("to start the second stream's work while the first is running")

Single Stream vs Multi-Stream Runtime Comparison
Matrix Size     Single (ms)        Multi (ms)         Speedup
-----------------------------------------------------------------
256             0.1390             0.2149             0.65x
512             0.3397             0.4055             0.84x
1024            1.7195             1.7618             0.98x
2048            9.3774             8.8402             1.06x
4096            65.0724            65.5416            0.99x

 What to observe:
Small matrices: streams may not help much
  → GPU is not the bottleneck, overhead dominates
Large matrices: streams should show meaningful speedup
  → Real compute and memory resources can be overlapped
If GPU is already 100% saturated: speedup ≈ 1.0x
  → No idle hardware to fill with second stream's work
